# Module 4: Synchronous vs. Asynchronous Execution

Welcome! Today we will learn about a critical behavior of GPUs: **asynchronous execution**.

**Section Goals:**
* Learn the difference between synchronous (blocking) and asynchronous (non-blocking) work.
* Understand the "Coffee Shop" analogy.
* See why timing GPU code without synchronization is a trap.

### Analogy: Ordering Coffee

* **Synchronous (Blocking):** You order a coffee, and you must stand at the counter blocking the line until the barista brews it and hands it to you. You can't do anything else.
* **Asynchronous (Non-blocking):** You order a coffee, get a ticket with order #42, and sit down to read a book. The barista prepares the coffee in the background. You only pause your book when they call your number.

### Visualizing Async Queue

Here is the coffee shop layout representing blocking vs. non-blocking operations:

![Coffee Shop](images/coffee-ordering-async.svg)

### Step 1: The Asynchronous Trap (No Sync)

When you run PyTorch GPU operations, the CPU enqueues the math task onto the GPU's execution queue (stream) and immediately moves to the next line of your Python script without waiting. Let's see what happens if we time this without synchronizing.

In [ ]:
x = torch.randn(5000, 5000, device="cuda")
t0 = time.perf_counter()  # Start CPU timer
y = torch.matmul(x, x)  # Submit Matmul to GPU queue
elapsed = time.perf_counter() - t0  # Stop timer immediately
print(f"Naive timed: {elapsed:.6f}s (Trap!)")

### Step 2: The Honest Measurement (With Sync)

To measure real GPU execution time, we must tell the CPU to wait until the GPU queue is completely empty. We do this using `torch.cuda.synchronize()`.

In [ ]:
torch.cuda.synchronize()  # Let previous tasks finish
t0 = time.perf_counter()  # Start CPU timer
y = torch.matmul(x, x)  # Submit Matmul to GPU
torch.cuda.synchronize()  # BLOCK CPU until GPU is done!
elapsed = time.perf_counter() - t0  # Stop timer
print(f"Honest timed: {elapsed:.6f}s (True time!)")

### Interpretation

Without synchronization, the time measured was almost 0. That's because we only timed how long it took the CPU to *enqueue* the task. With synchronization, we blocked the CPU until the GPU finished the actual calculation. Always synchronize when benchmarking!

### Module 4 Recap

* GPU operations are **asynchronous** (non-blocking) from the CPU's perspective.
* The CPU submits a task to the queue and runs ahead.
* **`torch.cuda.synchronize()`** acts as a barrier, forcing the CPU to wait until the GPU is done.